# 2ndBrain Purpose-Trained Laya Fine-Tune & CPU INT8 ONNX Export (Kaggle 2xT4 / Colab)

Upload `train.jsonl`, `val.jsonl`, and `calibration.jsonl` produced by `training/build_dataset.py`.

In [ ]:
!pip install -q torch transformers safetensors huggingface_hub onnx onnxruntime numpy

In [ ]:
import os, json
from pathlib import Path
from huggingface_hub import snapshot_download, hf_hub_download

BASE_CKPT = snapshot_download('convaiinnovations/laya-typed-decisions')
LAYA_CODE = snapshot_download('convaiinnovations/laya', allow_patterns=['rl_common.py', 'rl_agent_api.py'])
import sys; sys.path.insert(0, LAYA_CODE)
print('Base checkpoint:', BASE_CKPT)

In [ ]:
import random, torch
from safetensors.torch import load_file, save_file
from transformers import AutoTokenizer
from rl_common import encode_record, collate_items, build_model, proper_reward, seed_all

seed_all(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = json.loads((Path(BASE_CKPT) / 'rl_agent_config.json').read_text())
tok = AutoTokenizer.from_pretrained(str(Path(BASE_CKPT) / 'tokenizer'))
model = build_model(cfg, encoder_dir=str(Path(BASE_CKPT) / 'encoder')).to(device)
model.load_state_dict(load_file(str(Path(BASE_CKPT) / 'model.safetensors')), strict=False)
model.train()

train_rows = [json.loads(l) for l in Path('train.jsonl').read_text().splitlines() if l.strip()]
rng = random.Random(42)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

for epoch in range(3):
    rng.shuffle(train_rows)
    for i in range(0, len(train_rows), 8):
        batch_items = [encode_record(r, tok, cfg, rng, train=True) for r in train_rows[i:i+8]]
        b = collate_items(batch_items, tok.pad_token_id)
        if b is None: continue
        logits, _ = model(b['input_ids'].to(device), b['attention_mask'].to(device),
                          b['marker_pos'].to(device), b['marker_mask'].to(device), b['qtype'].to(device))
        probs = torch.softmax(logits, dim=-1)
        loss = -proper_reward(probs, b['target'].to(device), b['qtype'].to(device), b['marker_mask'].to(device)).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    print(f'Epoch {epoch+1} done, last loss={loss.item():.4f}')

OUT_CKPT = Path('laya_finetuned_ckpt')
OUT_CKPT.mkdir(exist_ok=True)
import shutil
for sub in ('encoder', 'tokenizer', 'rl_agent_config.json', 'config.json'):
    src = Path(BASE_CKPT) / sub
    dst = OUT_CKPT / sub
    if src.is_dir(): shutil.copytree(src, dst, dirs_exist_ok=True)
    elif src.exists(): shutil.copy2(src, dst)
save_file(model.state_dict(), str(OUT_CKPT / 'model.safetensors'))

In [ ]:
# Run export_int8_onnx.py to emit model_int8.onnx + signed model_manifest.json
# !python export_int8_onnx.py --ckpt-dir laya_finetuned_ckpt --calibration-jsonl calibration.jsonl --out-dir laya_2ndbrain_bundle --choice-labels domain_a,domain_b
# Copy laya_2ndbrain_bundle/ into %LOCALAPPDATA%\2ndBrain\state\models\ and run `brain model swap --model-dir ...`